## 1 - Loading Processed Data

In this step, we load processed video dataset from the previously notebooks


In [1]:
import pandas as pd
import os

In [2]:
print("--- LOADING PROCESSED DATA ---")

load_path = "./processed_images/fei_images_final.csv"
fei_imgs = pd.read_csv(load_path)

print(f"{len(fei_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(fei_imgs.sample(5))

load_path = "./processed_images/celeb_frames.csv"
celeb_imgs = pd.read_csv(load_path)

print(f"{len(celeb_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(celeb_imgs.sample(5))

--- LOADING PROCESSED DATA ---
8054 images loaded!


,path,label,split,dataset,method,target,source
6903,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/dataset_celeb_pipeline/FEI_MORPHV2_DATASET/train/fake/M_121-11_85-11_C03_B30_W30_PA03_PM00_F00_ssd.png,1,train,FEI,C03,121,85
279,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/dataset_celeb_pipeline/FEI_MORPHV2_DATASET/train/fake/M_150-11_170-11_C16_B30_W30_PA16_PM00_F00_ssd.png,1,train,FEI,C16,150,170
6034,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/dataset_celeb_pipeline/FEI_MORPHV2_DATASET/train/fake/M_28-11_146-11_C01_B30_W30_PA01_PM00_F00_ssd.png,1,train,FEI,C01,28,146
102,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/dataset_celeb_pipeline/FEI_MORPHV2_DATASET/train/fake/M_133-11_181-11_C15_B30_W30_PA15_PM00_F00_ssd.png,1,train,FEI,C15,133,181
6059,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/dataset_celeb_pipeline/FEI_MORPHV2_DATASET/train/fake/M_64-11_113-11_C01_B30_W30_PA01_PM00_F00_ssd.png,1,train,FEI,C01,64,113


162255 images loaded!


,path,label,split,dataset,method,target,source
69105,CELEBDFV3_DATASET/val/fake/id34_id31_GHOST_id31_id34_0004_f0_ssd.jpg,1,val,Celeb-DF-v3,GHOST,id34,id31
121878,CELEBDFV3_DATASET/test/fake/id01822_id2_EDTalk_id2_0002_test_id01822_GK2R7qog3Hc_f0_ssd.jpg,1,test,Celeb-DF-v3,EDTalk,id01822,id2
143528,CELEBDFV3_DATASET/train/fake/id04862_id56_IP_LAP_id56_0000_test_id04862_oKjLVc6gbbo_f2_ssd.jpg,1,train,Celeb-DF-v3,IP_LAP,id04862,id56
20319,CELEBDFV3_DATASET/train/fake/id11_id10_LIA_id10_id11_0004_f0_ssd.jpg,1,train,Celeb-DF-v3,LIA,id11,id10
128490,CELEBDFV3_DATASET/train/fake/id08374_id16_FLOAT_id16_0010_test_id08374_dGsdB4t_-ww_f0_ssd.jpg,1,train,Celeb-DF-v3,FLOAT,id08374,id16


## 2 - Master Dataset Compilation & Data Export

In this final preprocessing step, we consolidate our distinct datasets into standardized structures:

1. **Format Standardization:** We unify the column structure (`path`, `label`, `split`, `dataset`, `method`) across all dataframes and explicitly cast the labels into standard integers (`0` for Real, `1` for Fake).
2. **Master Dataset Assembly (FEI + FF++):** We concatenate the FEI and FaceForensics++ dataframes into a single, shuffled dataset. This `master_df` is exported as `master_dataset.csv` and contains our primary **Train**, **Validation**, and **Internal Test** splits.
3. **External Test Isolation (Celeb-DF):** The Celeb-DF dataset is deliberately excluded from the master dataset. It is held out completely as an **External Test Set**, serving exclusively for evaluating the model's cross-dataset generalization capabilities.
4. **Final Audit:** We output grouped distribution tables and random samples to verify the final dataset composition, split proportions, and label balancing.

In [3]:
# Prepare FEI
df_fei = fei_imgs[['path', 'label', 'split', 'method', 'target', 'source']].copy()
df_fei['dataset'] = 'FEI'
df_fei['label'] = df_fei['label'].replace({'original': 0, 'fake': 1}).astype(int)

# Prepare CELEB
df_celeb = celeb_imgs[['path', 'label', 'split', 'method', 'target', 'source']].copy()
df_celeb['dataset'] = 'Celeb-DF'
df_celeb['label'] = df_celeb['label'].astype(int)

# Merge
master_df = pd.concat([df_fei, df_celeb], ignore_index=True)
master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True)
master_df.to_csv("master_dataset.csv", index=False)

print("Merge completed! File saved as 'master_dataset.csv'.")

print("\n--- MASTER DATASET DISTRIBUTION (FEI + CELEB) ---")
distribution_master = master_df.groupby(['dataset', 'method', 'split', 'label']).size().unstack(fill_value=0)

if len(distribution_master.columns) == 2:
    distribution_master.columns = ['0 (Real)', '1 (Fake)']
display(distribution_master)

print("Master Dataset Sample:")
display(master_df.sample(5))

Merge completed! File saved as 'master_dataset.csv'.

--- MASTER DATASET DISTRIBUTION (FEI + CELEB) ---


0 (Real)  1 (Fake)
dataset  method    split                    
Celeb-DF AniTalker test          0       267
                   train         0      8235
                   val           0       348
         BlendFace test          0       438
                   train         0      4764
...                            ...       ...
FEI      C16       train         0      1046
                   val           0        30
         original  test         30         0
                   train       140         0
                   val          30         0

[93 rows x 2 columns]

Master Dataset Sample:


,path,label,split,method,target,source,dataset
115344,CELEBDFV3_DATASET/train/fake/id32_id34_GHOST_id34_id32_0009_f1_ssd.jpg,1,train,GHOST,id32,id34,Celeb-DF
71584,CELEBDFV3_DATASET/train/fake/id01228_id46_EchoMimic_id46_0004_test_id01228_555V_nuzxtA_f2_ssd.jpg,1,train,EchoMimic,id01228,id46,Celeb-DF
30170,CELEBDFV3_DATASET/test/fake/id16_id3_HyperReenact_id3_id16_0004_f1_ssd.jpg,1,test,HyperReenact,id16,id3,Celeb-DF
106545,CELEBDFV3_DATASET/train/fake/id6_id4_Celeb-DF-v2_id4_id6_0007_f2_ssd.jpg,1,train,Celeb-DF-v2,id6,id4,Celeb-DF
94979,CELEBDFV3_DATASET/train/fake/id35_id28_Celeb-DF-v2_id28_id35_0005_f1_ssd.jpg,1,train,Celeb-DF-v2,id35,id28,Celeb-DF


## 3 - Saving Data & Exporting Archive

To conclude this notebook, we first save our fully cleaned and processed DataFrames to local CSV files (e.g., `master_dataset.csv`). This ensures our prepared metadata is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the intensive preprocessing and extraction steps.

Finally, we package the entire structured dataset into a single, highly portable archive (`deepfake_dataset.zip`). This makes it easy to download, store, or transfer the data for model training.

**Contents of the final archive:**
* `FEI_MORPHV2_DATASET/`: The processed and split FEI dataset frames.
* `CELEBDFV3_DATASET/`: The processed and split Celeb-DF dataset frames.
* `master_dataset.csv`: The unified metadata for the training, validation, and test sets.

*(Note: We use the `-r` flag to include all subdirectories recursively and the `-q` flag to run the compression quietly, keeping the notebook output clean).*

In [4]:
print("--- SAVING DATAFRAME ---")

save_path_master = "master_dataset.csv"

master_df.to_csv(save_path_master, index=False)

print(f"Master Data successfully saved to: {save_path_master}")

--- SAVING DATAFRAME ---
Master Data successfully saved to: master_dataset.csv


In [5]:
!zip -rq deepfake_dataset.zip  FEI_MORPHV2_DATASET CELEBDFV3_DATASET master_dataset.csv